# 第6章：Fine-tuning for Classification

**目标：** 将预训练好的 GPT 模型改造为**垃圾短信分类器**，学习分类微调的完整流程

```
微调类型概览 → 准备数据集 → 创建 DataLoader → 加载预训练权重 → 添加分类头 → 训练 → 评估 → 使用
```

**前置回顾（第2-5章）：**
- 第2章：文本 → Token IDs → DataLoader
- 第3章：Causal Multi-Head Attention
- 第4章：完整 GPT 模型架构
- 第5章：预训练 + 加载 GPT-2 权重 + 文本生成
- 现在的问题：预训练模型只会「续写」文本，**不能做分类** → **需要微调！**

---

## 6.1 微调的类型 ⭐

LLM 最常见的两种微调方式：

| 类型 | 目标 | 输出 | 代表应用 |
|------|------|------|----------|
| **分类微调（Classification）** | 让模型输出固定的类别标签 | `spam` / `not spam` | 情感分析、垃圾邮件检测 |
| **指令微调（Instruction）** | 让模型按指令生成文本 | 自由文本 | ChatGPT 式对话 |

**本章聚焦分类微调：**
- 分类微调 = 给模型加一个「分类头」，输出**固定数量的类别**
- 模型只能预测训练过程中见过的类别（例如 spam / not spam）
- 相比指令微调，分类模型更加**专注**，通常也更容易训练

> 💡 **关键洞察：** 分类微调类似于用 CNN 做手写数字分类——都是替换输出层，冻结骨干网络，然后用有标签数据训练。

指令微调将在**第7章**详细讨论。

第5章让模型学会「说人话」（预测下一个词）。第6章在这个基础上微调，让它学会「做判断」（分类）。

spam = 垃圾短信（不想看），ham = 正常短信（想看）。这是 SMS 数据集的命名习惯。第6章的任务就是训练模型区分 spam 和 ham。

有了分辨能力——不只是机械地接话，而是能判断“这句话的意图是好的还是坏的”。


---

## 6.2 准备数据集 ⭐⭐

我们使用 **SMS Spam Collection** 数据集——包含 5,572 条短信，标注为 `ham`（正常）或 `spam`（垃圾）。

**数据处理流程：**
1. 下载并解压数据
2. 加载为 pandas DataFrame
3. 平衡类别（欠采样多数类）
4. 转换标签：`ham → 0`，`spam → 1`
5. 划分训练/验证/测试集（70%/10%/20%）

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import os

from transformers import GPT2Tokenizer

# ✅ 关键改动：指向本地文件夹，不联网
local_tokenizer_path = r"D:\Learning_buildLLM\local_tokenizer"
# 或者写绝对路径: "D:/Learning_buildLLM/local_tokenizer"

tokenizer = GPT2Tokenizer.from_pretrained(local_tokenizer_path, local_files_only=True)

# 验证是否成功
print(f"✅ 本地加载成功！词汇表大小: {tokenizer.vocab_size}")
print(f"编码测试: {tokenizer.encode('Hello world')}")

# ─── 第4章的所有组件（直接复用）───────────────────────────────────

# GPT-2 Small (124M) 的超参数配置
GPT_CONFIG_124M = {
    "vocab_size": 50257,      # BPE 词汇表大小
    "context_length": 1024,   # 最大序列长度（位置编码上限）
    "emb_dim": 768,           # 嵌入维度 / 隐藏层维度
    "n_heads": 12,            # 多头注意力的头数（每头 768/12=64 维）
    "n_layers": 12,           # Transformer Block 堆叠层数
    "drop_rate": 0.1,         # Dropout 概率
    "qkv_bias": False,        # Q/K/V 线性投影是否带偏置项
}


class MultiHeadAttention(nn.Module):
    """多头因果自注意力（第3章）"""
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out 必须能被 num_heads 整除"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout  = nn.Dropout(dropout)

        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(
            self.mask[:num_tokens, :num_tokens].bool(), -torch.inf
        )
        attn_weights = F.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vecs = attn_weights @ values
        context_vecs = context_vecs.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        context_vecs = self.out_proj(context_vecs)
        return context_vecs


class LayerNorm(nn.Module):
    """层归一化（第4章）"""
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    """GELU 激活函数（第4章）"""
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    """前馈网络（第4章）"""
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    """Transformer Block（第4章）"""
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    """完整 GPT 模型（第4章）"""
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


def generate_text_simple(model, idx, max_new_tokens, context_size):
    """贪心解码文本生成（第4章）"""
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


# ─── 第5章的权重加载工具 ─────────────────────────────────────────

def assign(left, right):
    """将 numpy 权重赋值给 PyTorch 参数"""
    if left.shape != right.shape:
        raise ValueError(f"Shape 不匹配: {left.shape} vs {right.shape}")
    return nn.Parameter(torch.tensor(right))


def load_weights_into_gpt(gpt, params):
    """将 Hugging Face GPT-2 权重加载到我们的 GPTModel 中（第5章）"""
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte.weight"])
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe.weight"])

    for b in range(len(gpt.trf_blocks)):
        q_w, k_w, v_w = np.split(
            params[f"h.{b}.attn.c_attn.weight"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T
        )
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T
        )
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T
        )

        q_b, k_b, v_b = np.split(
            params[f"h.{b}.attn.c_attn.bias"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b
        )
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b
        )
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b
        )

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params[f"h.{b}.attn.c_proj.weight"].T
        )
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params[f"h.{b}.attn.c_proj.bias"]
        )

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params[f"h.{b}.mlp.c_fc.weight"].T
        )
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params[f"h.{b}.mlp.c_fc.bias"]
        )
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params[f"h.{b}.mlp.c_proj.weight"].T
        )
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params[f"h.{b}.mlp.c_proj.bias"]
        )

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params[f"h.{b}.ln_1.weight"]
        )
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params[f"h.{b}.ln_1.bias"]
        )
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params[f"h.{b}.ln_2.weight"]
        )
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params[f"h.{b}.ln_2.bias"]
        )

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte.weight"])


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    return torch.tensor(encoded).unsqueeze(0)


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())


# ─── 初始化 ────────────────────────────────────────────────────
tokenizer = tiktoken.get_encoding("gpt2")
print("第4-5章组件加载完毕 ✓")

d:\Learning_buildLLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 本地加载成功！词汇表大小: 50257
编码测试: [15496, 995]
第4-5章组件加载完毕 ✓


### 下载并加载 SMS Spam 数据集

In [5]:
import urllib.request
import zipfile
from pathlib import Path
import pandas as pd

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} 已存在，跳过下载。")
        return

    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"文件已下载并保存为 {data_file_path}")


download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

sms_spam_collection\SMSSpamCollection.tsv 已存在，跳过下载。


In [6]:
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
print(f"→ 数据集大小: {len(df)} 条短信")
print(f"→ 类别分布:\n{df['Label'].value_counts()}")
df.head()

→ 数据集大小: 5572 条短信
→ 类别分布:
Label
ham     4825
spam     747
Name: count, dtype: int64


,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### 平衡数据集

数据集中 ham（正常）远多于 spam（垃圾），我们通过**欠采样（undersampling）** 使两类数量相等。

In [7]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df


balanced_df = create_balanced_dataset(df)
print(f"→ 平衡后类别分布:\n{balanced_df['Label'].value_counts()}")

→ 平衡后类别分布:
Label
ham     747
spam    747
Name: count, dtype: int64


In [8]:
# 将字符串标签转为整数：ham → 0, spam → 1
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})
balanced_df.head()

,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?


### 划分训练/验证/测试集

In [9]:
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    return train_df, validation_df, test_df


train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

print(f"→ 训练集: {len(train_df)} | 验证集: {len(validation_df)} | 测试集: {len(test_df)}")

→ 训练集: 1045 | 验证集: 149 | 测试集: 300


### ✏️ 练习

1. 查看 `balanced_df` 中最长和最短的短信各有多少个词？多少个 token？
2. 如果不做欠采样，直接用不平衡的数据训练，模型会出现什么问题？
3. **进阶**：除了欠采样，还有哪些处理类别不平衡的方法？试着搜索 `imbalanced-learn` 库。

In [10]:
1.
# 在 balanced_df 中统计
# 1. 计算每条短信的单词数 (用空格分割)
balanced_df['num_words'] = balanced_df['Text'].str.split().str.len()

# 2. 计算每条短信的 token 数 (使用你的 tokenizer)
balanced_df['num_tokens'] = balanced_df['Text'].apply(lambda x: len(tokenizer.encode(x)))

# 查看结果
print(f"单词数统计:\n{balanced_df['num_words'].describe()}\n")
print(f"Token数统计:\n{balanced_df['num_tokens'].describe()}\n")

# 找出最长和最短的具体例子
longest_word_row = balanced_df.loc[balanced_df['num_words'].idxmax()]
shortest_word_row = balanced_df.loc[balanced_df['num_words'].idxmin()]

print(f"单词数最多的短信 (词数: {longest_word_row['num_words']}):\n{longest_word_row['Text']}\n")
print(f"单词数最少的短信 (词数: {shortest_word_row['num_words']}):\n{shortest_word_row['Text']}\n")

longest_token_row = balanced_df.loc[balanced_df['num_tokens'].idxmax()]
shortest_token_row = balanced_df.loc[balanced_df['num_tokens'].idxmin()]

print(f"Token数最多的短信 (Token数: {longest_token_row['num_tokens']}):\n{longest_token_row['Text']}\n")
print(f"Token数最少的短信 (Token数: {shortest_token_row['num_tokens']}):\n{shortest_token_row['Text']}")

2.
#不平衡的话容易取巧

#模型会“偷懒”
#模型会找到一个“取巧”的策略：因为它发现只要把所有短信都判断为 ham（正常），就能获得 86.6% 的准确率。
# 它根本不需要去学习如何识别 spam，就能得到看起来很不错的分数。

#对少数类（spam）失效
#对于真正重要的任务——即识别出垃圾短信，模型的表现会非常差。因为它几乎没有学到关于 spam 的有效特征。
# 在我们关心的垃圾短信识别场景中，漏掉一条垃圾短信的代价，远高于把一条正常短信误判为垃圾短信。

#评估指标失真
#正如上面所说，准确率（Accuracy） 在这种情况下会变得毫无意义。
# 一个模型即使永远不输出 spam，也能获得很高的准确率，但它完全不符合我们的使用要求。
# 因此，在类别不平衡时，我们需要使用 精确率（Precision）、召回率（Recall）、F1分数（F1-Score） 或 AUC曲线 等更科学的指标来评估模型。

3.
#过采样 (Oversampling)

#原理：通过复制或生成新的样本来增加少数类（spam）的数量。

#简单方法：随机过采样（Random Oversampling），即随机复制已有的少数类样本。这种方法简单，但容易导致模型对复制的样本过拟合。

#高级方法：SMOTE (Synthetic Minority Over-sampling Technique)。它不是简单复制，而是在少数类样本之间进行“插值”，
# 生成全新的、合成的样本，这能有效缓解过拟合问题。

#imbalanced-learn这个库里面有这个   
#欠采样	减少多数类（ham）样本，类似我们做的，但算法更聪明。	RandomUnderSampler (我们做的)、TomekLinks
#过采样	增加少数类（spam）样本，不只是复制，还会“创造”新样本。	SMOTE、ADASYN
#组合采样	把过采样和欠采样结合起来用。	SMOTETomek、SMOTEENN

    



单词数统计:
count    1494.000000
mean       19.023427
std        10.117732
min         1.000000
25%        10.000000
50%        21.000000
75%        27.000000
max        95.000000
Name: num_words, dtype: float64

Token数统计:
count    1494.000000
mean       30.590361
std        17.222886
min         2.000000
25%        14.000000
50%        33.000000
75%        44.000000
max       120.000000
Name: num_tokens, dtype: float64

单词数最多的短信 (词数: 95):
Hey sweet, I was wondering when you had a moment if you might come to me ? I want to send a file to someone but it won't go over yahoo for them because their connection sucks, remember when you set up that page for me to go to and download the format disc ? Could you tell me how to do that ? Or do you know some other way to download big files ? Because they can download stuff directly from the internet. Any help would be great, my prey ... *teasing kiss*

单词数最少的短信 (词数: 1):
Ok..

Token数最多的短信 (Token数: 120):
Storming msg: Wen u lift d phne, u say "HELLO" Do 

3.0

---

## 6.3 创建 DataLoader ⭐⭐

不同短信长度不一，需要**填充（padding）**到统一长度才能组成 batch。

**策略：**
- 用 `<|endoftext|>`（token ID = 50256）作为 padding token
- 所有序列填充到训练集中最长的序列长度
- 超过最大长度的序列会被截断

### 短信预处理流程：分词 → 截断 → 补位

假设 `max_length = 10`，`pad_token_id = 50256`

### 示例1：短短信（"Hi"）

| 步骤 | 操作 | 结果 |
| :--- | :--- | :--- |
| 原始 | `"Hi"` | `"Hi"` |
| 1. 分词 | `tokenizer.encode("Hi")` | `[123, 456]`（2个token）|
| 2. 截断 | 2 ≤ 10，不需要截断 | `[123, 456]` |
| 3. 补位 | 补 8 个 50256 | `[123, 456, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]` |

### 示例2：长短信（15个token）

| 步骤 | 操作 | 结果 |
| :--- | :--- | :--- |
| 原始 | 很长的短信... | 假设分词后有 15 个 token |
| 1. 分词 | `tokenizer.encode(long_text)` | `[a, b, c, d, e, f, g, h, i, j, k, l, m, n, o]`（15个）|
| 2. 截断 | 只取前 10 个 | `[a, b, c, d, e, f, g, h, i, j]` |
| 3. 补位 | 不需要补（刚好 10 个）| `[a, b, c, d, e, f, g, h, i, j]` |

> **顺序固定：先分词 → 再截断 → 最后补位。保证每条短信最终长度都是 `max_length`。**

In [16]:
from torch.utils.data import Dataset, DataLoader

# 1️⃣ Dataset：定义数据长什么样，如何预处理
class SpamDataset(Dataset):
    #CSV = Comma-Separated Values（逗号分隔值）就是一种表格文件，可以用 Excel 打开，也可以用代码读取。
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        ## 1. 读取 CSV，每条短信单独分词
        self.data = pd.read_csv(csv_file)

        # 预先分词
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        # 2. 如果没指定 max_length，就用最长那条的长度
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
        # 2b. 截断：超过 max_length 的切掉后面部分
            # 截断超长序列
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]
#统一长度的实现：先截断超过 max_length 的，再用 pad_token_id 填充不足的。

        # 3. 填充：不足 max_length 的用 pad_token_id 补到一样长
        # 填充到统一长度
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        return max(len(encoded_text) for encoded_text in self.encoded_texts)

In [21]:
# 2️⃣ 创建 Dataset 实例（这里才开始真正读数据、预处理）
# 创建三个数据集 训练集、验证集、测试集，每个都是已经处理好的数据集合
train_dataset = SpamDataset(csv_file="train.csv", max_length=None, tokenizer=tokenizer)
print(f"→ 训练集最长序列: {train_dataset.max_length} tokens")

val_dataset = SpamDataset(
    csv_file="validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer
)

→ 训练集最长序列: 120 tokens


#Dataset 已经处理好了数据，DataLoader 负责在训练时「按需取用」。

#### DataLoader 的核心作用

| 功能 | 说明 | 代码体现 |
| :--- | :--- | :--- |
| 批量取数据 | 一次取 8 条，不是 1 条 | `batch_size=8` |
| 打乱顺序 | 每轮训练顺序不同 | `shuffle=True` |
| 多线程加载 | 边训练边准备下一批 | `num_workers` |

In [ ]:
# 3️⃣ DataLoader：把 Dataset 包装成批量加载器
# 创建 DataLoader
num_workers = 0  #用几个进程加载（0=单进程）
batch_size = 8  #一次取 8 条短信

torch.manual_seed(123)

#训练、验证、测试集
train_loader = DataLoader(
    dataset=train_dataset, batch_size=batch_size,
    shuffle=True, num_workers=num_workers, drop_last=True,
)
val_loader = DataLoader(
    dataset=val_dataset, batch_size=batch_size,
    num_workers=num_workers, drop_last=False,
)
test_loader = DataLoader(
    dataset=test_dataset, batch_size=batch_size,
    num_workers=num_workers, drop_last=False,
)

# 验证 batch 维度
for input_batch, target_batch in train_loader:
    pass
print(f"→ 输入 batch: {input_batch.shape}")
print(f"→ 标签 batch: {target_batch.shape}")
print(f"→ {len(train_loader)} 训练 batches | {len(val_loader)} 验证 batches | {len(test_loader)} 测试 batches")

→ 输入 batch: torch.Size([8, 120])
→ 标签 batch: torch.Size([8])
→ 130 训练 batches | 19 验证 batches | 38 测试 batches


验证 batch 就是「先试吃一口」：确认数据形状正确、没有异常，再正式开训练。避免跑了几十个小时才发现数据格式有问题。

### ✏️ 练习

1. 改变 `batch_size`（如 4、16、32），观察 batch 数量的变化。更大的 batch_size 有什么利弊？
2. 如果不做 padding，直接用变长序列训练，会遇到什么问题？
3. **进阶**：当前我们把所有序列 pad 到数据集最长长度。能否改为 pad 到**每个 batch 的最长长度**？这样做有什么好处？

矩阵是用来批量计算的「表格」。文本通过 token ID → Embedding → Transformer → 分类头，每一步都是矩阵运算。第6章的处理和第5章几乎一样，只是把输出层从「预测下一个词」改成了「二分类 spam/ham」。

In [ ]:
# 在这里做实验
import sys
print(sys.executable)

#batch_size 越大，batch 数量越少。

#batch_size	优点	缺点
#小（4-8）	省显存、梯度更新频繁	训练慢、梯度噪声大
#大（32-64）	训练快、梯度稳定	需要大显存、可能收敛到局部最优

2.
#PyTorch 要求一个 batch 内的所有张量形状相同。矩阵计算必须要这些要求才能批量计算才能效率高
#会报错

#其他问题：

#问题	说明
#无法批量化	不能打包成矩阵，只能一条一条算（极慢）
#模型无法处理	模型输入维度固定，变长无法对齐
#内存碎片	变长序列难以高效存储


3.
#会更好

def collate_fn(batch):
    """自定义批处理函数：动态 padding 到 batch 内最长长度"""
    texts, labels = zip(*batch)
    
    # 找出这个 batch 里的最长长度
    max_len = max(len(t) for t in texts)
    
    # 只 pad 到 max_len（不是全局最大）
    padded_texts = [t + [50256] * (max_len - len(t)) for t in texts]
    
    return torch.tensor(padded_texts), torch.tensor(labels)

# 使用
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,  # ← 自定义批处理
)
#好处：

#优势	说明
#减少计算浪费	短的 batch 不用 pad 到 50，只 pad 到 8（省显存）
#加速训练	每个 batch 长度更短，计算更快
#更灵活	适合长度差异大的数据集

##### 文本 → 模型的完整流程
```
原始文本: "WINNER!! Claim $1000"
    ↓ 分词
Token IDs: [123, 456, 789, 101, 112]（5个数字）
    ↓ 打包成 batch（和别的短信一起）
inputs.shape = (8, 50)   # 8条短信，每条50个token
    ↓ Token Embedding（第2章）
(8, 50, 768)   # 每个 token 变成 768 维向量
    ↓ 位置编码（第2章）
(8, 50, 768)   # 加上位置信息
    ↓ Transformer Block × N（第3-4章）
(8, 50, 768)   # 经过注意力、FFN
    ↓ 分类头（第6章新加的）
(8, 2)         # 8条短信 → 每条输出 spam/ham 的概率
```

---

## 6.4 加载预训练权重 ⭐⭐

我们复用第5章的做法，从 Hugging Face 下载 GPT-2 (124M) 的预训练权重并加载到模型中。

**关键配置变化：**
- `drop_rate` 设为 **0.0**（微调时不需要太多 dropout）
- `qkv_bias` 设为 **True**（GPT-2 原版带偏置）

In [26]:
from transformers import GPT2Model

# 指向你的本地模型文件夹
local_model_path = "D:/Learning_buildLLM/models/gpt2"

# 加载模型（不联网）
model_hf = GPT2Model.from_pretrained(local_model_path, local_files_only=True)
model_hf.eval()

print(f"✅ 模型加载成功！")
print(f"   配置: {model_hf.config}")
print(f"   参数量: {sum(p.numel() for p in model_hf.parameters()):,}")

def download_and_load_gpt2(model_size, models_dir):
    """从 Hugging Face 下载 GPT-2 权重"""
    model_hf = HF_GPT2Model.from_pretrained(model_size, cache_dir=models_dir)
    model_hf.eval()
    params = {}
    for name, param in model_hf.named_parameters():
        params[name] = param.detach().numpy()
    for name, buf in model_hf.named_buffers():
        params[name] = buf.detach().numpy()
    return params


CHOOSE_MODEL = "gpt2"

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,        # 微调时 dropout 设为 0
    "qkv_bias": True,        # GPT-2 原版带偏置
    "emb_dim": 768,
    "n_layers": 12,
    "n_heads": 12,
}

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"数据集长度 {train_dataset.max_length} 超过模型上下文长度 {BASE_CONFIG['context_length']}"
)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6418.81it/s]

✅ 模型加载成功！
   配置: GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.8.0",
  "use_cache": true,
  "vocab_size": 50257
}

   参数量: 124

In [27]:
import os
from transformers import GPT2Model

# 指向你的本地模型文件夹（你已经有完整文件的位置）
local_model_path = "D:/Learning_buildLLM/models/gpt2"

# 检查文件夹是否存在
if not os.path.exists(local_model_path):
    raise FileNotFoundError(f"模型文件夹不存在: {local_model_path}")

print(f"正在从本地加载 {CHOOSE_MODEL} 模型...")

# 直接从本地加载，不联网
model_hf = GPT2Model.from_pretrained(local_model_path, local_files_only=True)
model_hf.eval()

# 提取参数（和原来一样）
params = {}
for name, param in model_hf.named_parameters():
    params[name] = param.detach().numpy()
for name, buf in model_hf.named_buffers():
    params[name] = buf.detach().numpy()

print(f"加载完成！参数数量: {len(params)}")

# 加载到你自己的模型中
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()
print("\n预训练权重加载完毕 ✓")

正在从本地加载 gpt2 模型...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6727.76it/s]

加载完成！参数数量: 148



预训练权重加载完毕 ✓


### 验证权重：让模型生成文本

In [28]:
# 验证模型可以生成连贯文本
text = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)
print(f"→ 输入: {text}")
print(f"→ 生成: {token_ids_to_text(token_ids, tokenizer)}")

→ 输入: Every effort moves you
→ 生成: Every effort moves you forward.

The first step is to understand the importance of your work


### 试试用 prompt 做分类？

在微调之前，先看看预训练模型能否直接通过 prompt 来判断垃圾短信：

In [25]:
text = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with 'yes' or 'no': 'You are a winner


> 💡 **观察：** 预训练模型并不擅长「遵循指令」——它只会续写文本，不会回答 yes/no 问题。
> 这正是我们需要**微调**的原因！指令微调会在第7章介绍。

---

## 6.5 添加分类头 ⭐⭐⭐

这是本章的**核心步骤**：

1. **冻结所有参数** — 让预训练权重不被破坏
2. **替换 `out_head`** — 从 50,257 维（词汇表）→ 2 维（spam / not spam）
3. **解冻最后一层 Transformer Block + final_norm** — 适度微调提升效果

```
为什么用最后一个 token 做分类？

在 Causal Attention 中，每个 token 只能看到自己和之前的 token。
因此最后一个 token 拥有整个序列的信息 → 最适合做分类判断。
```

前 11 层：冻结（参数不动，保留语言能力）

第 12 层：解冻（训练适应分类）

final_norm：解冻

输出层：Linear(768, 2) ← 改成二分类

目标：判断 spam/ham


2 维是固定的，代表「两个类别」。标签是人为定义的（0=ham，1=spam）。预训练时没有标签，只有「下一个词」。

In [ ]:
# 第一步：冻结所有参数
for param in model.parameters():
    param.requires_grad = False

# 第二步：替换输出层 (50,257 → 2)  预测 spam/ham（2 个类别的概率）
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(          #这里是out_head 输出2维固定标签，用于判断而不是预测
    in_features=BASE_CONFIG["emb_dim"],
    out_features=num_classes
)

# 第三步：解冻最后一层 Transformer Block 和 final_norm
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

# 统计可训练参数
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"→ 总参数: {total_params:,}")
print(f"→ 可训练参数: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

→ 总参数: 124,441,346
→ 可训练参数: 7,090,946 (5.7%)


### 验证新的输出维度

In [31]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print(f"→ 输入: {inputs}, shape: {inputs.shape}")

with torch.no_grad():
    outputs = model(inputs)
print(f"→ 输出 shape: {outputs.shape}  (batch=1, seq=4, classes=2)")

# 取最后一个 token 的输出作为分类依据
print(f"→ 最后一个 token 的 logits: {outputs[:, -1, :]}")

→ 输入: tensor([[5211,  345,  423,  640]]), shape: torch.Size([1, 4])
→ 输出 shape: torch.Size([1, 4, 2])  (batch=1, seq=4, classes=2)
→ 最后一个 token 的 logits: tensor([[-3.5983,  3.9902]])


### ✏️ 练习

1. 如果只训练 `out_head`（不解冻最后一层 TransformerBlock），准确率会下降多少？
2. 如果解冻**所有**层（full fine-tuning），训练速度和效果会怎样？
3. **进阶**：搜索 LoRA（Low-Rank Adaptation），了解另一种参数高效微调方法。

In [ ]:
# 在这里做实验
1.
#感觉会下降很多，因为12层都是这样的，适应性判断会比较差

2.
#也会下降吧，都不训练语言了

3.
#让我搜搜





---

## 6.6 计算分类损失和准确率 ⭐⭐

分类微调的评估指标：
- **交叉熵损失（Cross-Entropy Loss）** — 可微分，用于训练
- **分类准确率（Accuracy）** — 直观，用于评估

与第5章预训练的区别：
- 预训练时，我们优化**所有位置**的 next-token prediction loss
- 分类微调时，我们只取**最后一个 token** 的输出做分类

$$\text{Prediction} = \arg\max(\text{model}(x)[:, -1, :])$$

In [32]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    """计算整个 data_loader 上的分类准确率"""
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)
            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # 最后一个 token
            predicted_labels = torch.argmax(logits, dim=-1)
            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples


def calc_loss_batch(input_batch, target_batch, model, device):
    """计算单个 batch 的交叉熵损失（只看最后一个 token）"""
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = F.cross_entropy(logits, target_batch)
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    """计算整个 data_loader 上的平均损失"""
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [33]:
# 选择设备
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"→ 使用设备: {device}")
model.to(device)

# 训练前的基线指标
torch.manual_seed(123)
with torch.no_grad():
    train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
    val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print(f"\n训练前基线指标:")
print(f"  训练准确率: {train_accuracy*100:.2f}%")
print(f"  验证准确率: {val_accuracy*100:.2f}%")
print(f"  训练损失: {train_loss:.3f}")
print(f"  验证损失: {val_loss:.3f}")

→ 使用设备: cpu

训练前基线指标:
  训练准确率: 46.25%
  验证准确率: 45.00%
  训练损失: 3.075
  验证损失: 2.583


#### 分类模型内部流程
```
输入: "WINNER!! Claim $1000"
↓
模型内部计算
↓
原始打分 (logits): [2.1, -1.3]
├── 索引0: spam 的分数
└── 索引1: ham 的分数
↓
Softmax 归一化
↓
概率分布 (probs): [0.97, 0.03]
├── 97% 概率是 spam
└── 3% 概率是 ham
├──→ 取 argmax → 最终答案: "spam" (给我们看的)
└──→ 查正确答案:
如果正确答案是 spam (索引0)
→ 损失 = -log(0.97) = 0.03 (很小 ✓)

如果正确答案是 ham (索引1)
→ 损失 = -log(0.03) = 3.5 (很大 ✗)

```
#### 核心理解

| 阶段 | 输出 | 作用 | 谁看 |
| :--- | :--- | :--- | :--- |
| 原始打分 | `[2.1, -1.3]` | 模型对各类别的“印象” | 内部计算 |
| 概率分布 | `[0.97, 0.03]` | 模型表达的“把握程度” | **损失函数**（学习信号）|
| 最终答案 | `"spam"` | 模型判断的类别 | **用户**（给我们看的）|

#### 损失的含义

> **损失 = -log(模型对正确答案的概率)**

- 正确答案概率 = 0.97 → 损失 = 0.03 ✅ 很小（自信）
- 正确答案概率 = 0.50 → 损失 = 0.69 ⚠️ 中等（犹豫）
- 正确答案概率 = 0.01 → 损失 = 4.61 ❌ 很大（错误）

**训练目标：让正确答案的概率 → 1，损失 → 0**

> 💡 **观察：** 训练前准确率大约在 50% 左右——相当于随机猜测。这很正常，因为分类头是随机初始化的。

---

## 6.7 训练分类器 ⭐⭐⭐

训练循环与第5章的 `train_model_simple` 基本相同，主要差异：
1. 跟踪 `examples_seen`（而非 `tokens_seen`）
2. 每个 epoch 后计算准确率（而非生成文本样本）

In [36]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                            num_epochs, eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # 每个 epoch 结束后计算准确率
        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [39]:
import time

# 🚀 开始训练！
start_time = time.time()

torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs = 5 #训练5轮

train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
print(f"\n训练完成！耗时 {(end_time - start_time) / 60:.2f} 分钟。")

Ep 1 (Step 000000): Train loss 0.632, Val loss 0.590
Ep 1 (Step 000050): Train loss 0.393, Val loss 0.446
Ep 1 (Step 000100): Train loss 0.319, Val loss 0.419
Training accuracy: 85.00% | Validation accuracy: 87.50%
Ep 2 (Step 000150): Train loss 0.609, Val loss 0.387
Ep 2 (Step 000200): Train loss 0.337, Val loss 0.313
Ep 2 (Step 000250): Train loss 0.364, Val loss 0.170
Training accuracy: 87.50% | Validation accuracy: 90.00%
Ep 3 (Step 000300): Train loss 0.214, Val loss 0.136
Ep 3 (Step 000350): Train loss 0.210, Val loss 0.076
Training accuracy: 97.50% | Validation accuracy: 97.50%
Ep 4 (Step 000400): Train loss 0.039, Val loss 0.082
Ep 4 (Step 000450): Train loss 0.082, Val loss 0.066
Ep 4 (Step 000500): Train loss 0.162, Val loss 0.069
Training accuracy: 97.50% | Validation accuracy: 97.50%
Ep 5 (Step 000550): Train loss 0.166, Val loss 0.082
Ep 5 (Step 000600): Train loss 0.049, Val loss 0.047
Training accuracy: 100.00% | Validation accuracy: 97.50%

训练完成！耗时 21.79 分钟。


### 可视化训练过程

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib as mpl

# ========== 重置所有 matplotlib 设置 ==========
mpl.rcParams.update(mpl.rcParamsDefault)

# ========== 设置中文字体 ==========
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows 黑体
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(6, 3.5))
    ax1.plot(epochs_seen, train_values, label=f"训练 {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"验证 {label}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    ax2 = ax1.twiny()
    ax2.plot(examples_seen, train_values, alpha=0)
    ax2.set_xlabel("样本数")

    fig.tight_layout()
    plt.show()


# 绘制损失曲线
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))
plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses, label="loss")

NameError: name 'num_epochs' is not defined

In [ ]:
# 绘制准确率曲线
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))
plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

### 最终评估

In [ ]:
# 在完整数据集上计算最终准确率
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"训练准确率: {train_accuracy*100:.2f}%")
print(f"验证准确率: {val_accuracy*100:.2f}%")
print(f"测试准确率: {test_accuracy*100:.2f}%")

> 💡 **观察：** 测试集准确率略低于训练/验证集，这说明模型存在轻微的过拟合。可以通过增大 `drop_rate` 或 `weight_decay` 来缓解。

### ✏️ 练习

1. 调整学习率（如 1e-4, 1e-5），观察训练曲线和最终准确率的变化。
2. 增加 `num_epochs` 到 10，观察是否出现过拟合（训练 loss 下降但验证 loss 上升）。
3. **进阶**：添加梯度裁剪（`torch.nn.utils.clip_grad_norm_`），看看对训练稳定性有什么影响。

In [ ]:
# 在这里做实验

---

## 6.8 使用分类器 ⭐

训练完成后，我们可以用模型对新短信进行分类：

In [2]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    """对一条文本进行 spam/not spam 分类"""
    model.eval()

    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]

    input_ids = input_ids[:min(max_length, supported_context_length)]
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0)

    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
    predicted_label = torch.argmax(logits, dim=-1).item()

    return "spam" if predicted_label == 1 else "not spam"

In [3]:
# 测试：垃圾短信
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)
print(f"\"...{text_1[:50]}...\" → {classify_review(text_1, model, tokenizer, device, max_length=train_dataset.max_length)}")

# 测试：正常短信
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)
print(f"\"...{text_2[:50]}...\" → {classify_review(text_2, model, tokenizer, device, max_length=train_dataset.max_length)}")

NameError: name 'model' is not defined

### 保存和加载模型

In [ ]:
# 保存微调后的模型
torch.save(model.state_dict(), "spam_classifier.pth")
print("模型已保存为 spam_classifier.pth")

# 加载方式示例（在新的 session 中使用）：
# model = GPTModel(BASE_CONFIG)
# model.out_head = torch.nn.Linear(in_features=768, out_features=2)
# model.load_state_dict(torch.load("spam_classifier.pth", map_location=device, weights_only=True))
# model.eval()

### ✏️ 练习

1. 自己编写几条短信（中文或英文），测试分类器的效果。
2. 模型对哪类短信容易判断错误？分析原因。
3. **进阶**：将保存的模型在新的 notebook 中加载并使用。

In [ ]:
# 在这里做实验

---

## 6.9 完整流程回顾

In [4]:
print("""
═══════════════════════════════════════════════════════════
          第 6 章：分类微调 — 完整流程
═══════════════════════════════════════════════════════════

  1. 了解微调类型（分类 vs 指令）
     ↓
  2. 下载并准备 SMS Spam 数据集
     ↓  平衡类别 → 划分 train/val/test
  3. 创建 SpamDataset + DataLoader
     ↓  tokenize → padding → batch
  4. 加载预训练 GPT-2 权重
     ↓  验证文本生成能力
  5. 添加分类头
     ↓  冻结 backbone → 替换 out_head → 解冻最后一层
  6. 计算分类损失和准确率
     ↓  cross_entropy(logits[:, -1, :], labels)
  7. 训练分类器
     ↓  5 epochs, AdamW, lr=5e-5
  8. 使用分类器对新文本做预测
     ↓  tokenize → pad → forward → argmax
  9. 第 7 章：指令微调 — 让模型学会对话！

═══════════════════════════════════════════════════════════

关键收获:
  • 分类微调 = 冻结 backbone + 替换输出头 + 用有标签数据训练
  • 只取最后一个 token 的输出做分类（因为 causal attention）
  • 解冻最后几层可以显著提升效果
  • 只需训练约 7% 的参数就能达到 ~95% 以上的准确率
""")


═══════════════════════════════════════════════════════════
          第 6 章：分类微调 — 完整流程
═══════════════════════════════════════════════════════════

  1. 了解微调类型（分类 vs 指令）
     ↓
  2. 下载并准备 SMS Spam 数据集
     ↓  平衡类别 → 划分 train/val/test
  3. 创建 SpamDataset + DataLoader
     ↓  tokenize → padding → batch
  4. 加载预训练 GPT-2 权重
     ↓  验证文本生成能力
  5. 添加分类头
     ↓  冻结 backbone → 替换 out_head → 解冻最后一层
  6. 计算分类损失和准确率
     ↓  cross_entropy(logits[:, -1, :], labels)
  7. 训练分类器
     ↓  5 epochs, AdamW, lr=5e-5
  8. 使用分类器对新文本做预测
     ↓  tokenize → pad → forward → argmax
  9. 第 7 章：指令微调 — 让模型学会对话！

═══════════════════════════════════════════════════════════

关键收获:
  • 分类微调 = 冻结 backbone + 替换输出头 + 用有标签数据训练
  • 只取最后一个 token 的输出做分类（因为 causal attention）
  • 解冻最后几层可以显著提升效果
  • 只需训练约 7% 的参数就能达到 ~95% 以上的准确率



---

## 📝 本章核心 Checklist

学完本章后，检查你是否能回答以下问题：

- [ ] 分类微调和指令微调有什么区别？各适用于什么场景？
- [ ] 为什么要用最后一个 token 的输出做分类？
- [ ] 为什么要冻结大部分参数？只训练哪些层？
- [ ] `out_head` 替换前后，模型的输出维度有什么变化？
- [ ] `SpamDataset` 中 padding 的作用是什么？为什么用 `<|endoftext|>` 做 padding token？
- [ ] 训练中的 `calc_loss_batch` 与第5章有什么区别？
- [ ] 怎样判断模型是否过拟合？有哪些缓解方法？
- [ ] 如何保存和加载微调后的模型？

全部能回答 → 进入**第 7 章：指令微调**——让模型学会像 ChatGPT 一样对话！

## ✅ Checklist 参考答案

### 1. 分类微调和指令微调有什么区别？各适用于什么场景？

| | 分类微调 | 指令微调 |
|---|---|---|
| **目标** | 输出固定类别标签 | 按指令生成自由文本 |
| **输出** | `spam` / `not spam` 等离散标签 | 任意长度的文本 |
| **输出层** | `Linear(768, num_classes)` | 保持 `Linear(768, 50257)` |
| **损失** | 分类交叉熵（仅最后一个 token） | 语言模型交叉熵（所有位置） |
| **代表应用** | 情感分析、垃圾邮件检测 | ChatGPT 式对话、问答 |

- 分类微调更**专注**，标签空间有限，通常更容易训练
- 指令微调更**通用**，能处理开放式任务，但训练更复杂

---

### 2. 为什么要用最后一个 token 的输出做分类？

GPT 使用 **Causal Attention**（因果注意力掩码），每个位置只能看到自己和之前的 token。因此：

- 第 1 个 token 只知道自己
- 第 2 个 token 知道第 1、2 个
- **最后一个 token 是唯一一个看到了序列中所有 token 信息的表示**

代码体现：`logits = model(input_batch)[:, -1, :]`

---

### 3. 为什么要冻结大部分参数？只训练哪些层？

**冻结原因：**
- 保留预训练学到的语言能力，不让小数据集破坏已有权重
- 减少训练参数，降低过拟合风险和计算开销

**三步策略：**

1. **冻结所有参数**：`for param in model.parameters(): param.requires_grad = False`
2. **替换 out_head**：`Linear(768, 2)` — 新层默认 `requires_grad=True`
3. **解冻最后一层 Transformer Block + final_norm**：
   ```python
   for param in model.trf_blocks[-1].parameters():
       param.requires_grad = True
   for param in model.final_norm.parameters():
       param.requires_grad = True
   ```

最终可训练参数仅占 **5.7%**（7,090,946 / 124,441,346）。

---

### 4. `out_head` 替换前后，模型的输出维度有什么变化？

| | 替换前（预训练） | 替换后（分类微调） |
|---|---|---|
| **定义** | `Linear(768, 50257)` | `Linear(768, 2)` |
| **输出 shape** | `(batch, seq_len, 50257)` | `(batch, seq_len, 2)` |
| **含义** | 每个位置输出词表上所有 token 的概率 | 每个位置输出 2 个类别的 logits |

例如输入 4 个 token → 输出 shape 从 `(1, 4, 50257)` 变为 `(1, 4, 2)`，取最后一个 token 的 logits `tensor([[-3.60, 3.99]])` 做分类。

---

### 5. `SpamDataset` 中 padding 的作用是什么？为什么用 `<|endoftext|>` 做 padding token？

**Padding 的作用：** 不同短信长度不一，PyTorch 要求同一 batch 内张量形状相同，因此需要填充到统一长度。

**为什么用 `<|endoftext|>`（token ID = 50256）：**
- 它是 GPT-2 的特殊 token，词汇表的最后一个
- 预训练时用于分隔不同文档，模型已学会在此 token 处"重置"上下文
- 用它做 padding，模型不会将 padding 信息混入有效内容的表示
- 无需引入新的 padding token，复用已有特殊 token

**处理流程：** 分词 → 截断（超过 max_length 切掉）→ 补位（不足的用 50256 补齐）

---

### 6. 训练中的 `calc_loss_batch` 与第5章有什么区别？

| | 第5章（预训练） | 第6章（分类微调） |
|---|---|---|
| **logits 取法** | 所有位置：`model(input_batch)` | 仅最后一个 token：`model(input_batch)[:, -1, :]` |
| **损失函数** | 语言模型交叉熵（目标 = token ID 序列） | 分类交叉熵 `F.cross_entropy(logits, target_batch)`（目标 = 类别标签） |
| **训练循环指标** | `tokens_seen` | `examples_seen` |
| **评估方式** | 生成文本样本 | 计算准确率 |

---

### 7. 怎样判断模型是否过拟合？有哪些缓解方法？

**判断方法：**
1. **训练损失持续下降，验证损失开始上升** — 最核心的信号
2. **训练准确率远高于验证/测试准确率**
3. **观察损失/准确率曲线**，验证曲线反向变化即为过拟合

**缓解方法：**
1. 增大 `drop_rate`（当前为 0.0，可设为 0.1）
2. 增大 `weight_decay`（当前为 0.1）
3. 减少 `num_epochs` / 提前停止（Early Stopping）
4. 梯度裁剪 `torch.nn.utils.clip_grad_norm_`
5. 冻结更多层，减少可训练参数量
6. 数据增强

---

### 8. 如何保存和加载微调后的模型？

**保存：**
```python
torch.save(model.state_dict(), "spam_classifier.pth")
```

**加载：**
```python
# 1. 先重建相同结构的模型
model = GPTModel(BASE_CONFIG)
model.out_head = torch.nn.Linear(in_features=768, out_features=2)
# 2. 加载权重
model.load_state_dict(torch.load("spam_classifier.pth", map_location=device, weights_only=True))
model.eval()
```

**关键细节：**
- `state_dict()` 只保存参数字典，不保存模型结构 → 加载时必须先重建模型
- 加载前必须先替换 `out_head` 为 `Linear(768, 2)`，否则维度不匹配
- `map_location=device` 确保在目标设备上加载
- `weights_only=True` 是安全选项，只加载张量数据
- 加载后调用 `model.eval()` 切换到评估模式